### PySpark vs Pandas Comparison
* **PySpark**: Distributed processing, handles big data, lazy evaluation, DataFrame API similar to SQL, runs on clusters.
* **Pandas**: In-memory processing, best for small/medium data, eager evaluation, DataFrame API, runs on single machine.
* Use PySpark for scalability and distributed analytics; use Pandas for fast prototyping and local analysis.

### Joins in PySpark
* **Inner Join**: Returns rows with matching keys in both DataFrames.
* **Left Join**: Returns all rows from left DataFrame, matched rows from right.
* **Right Join**: Returns all rows from right DataFrame, matched rows from left.
* **Outer Join**: Returns all rows from both DataFrames, matched where possible.
* Syntax: `df1.join(df2, on="key", how="inner")`

### Window Functions in PySpark
* Used for calculations across a set of rows related to the current row (e.g., running totals, rankings).
* Common functions: `row_number()`, `rank()`, `sum() over window`, `avg() over window`.
* Example: Calculate running total of sales per user.

### User-Defined Functions (UDFs)
* Custom Python functions applied to DataFrame columns.
* Register with `F.udf()` and use in `withColumn()`.
* Useful for complex transformations not covered by built-in functions.

#### 1. Load full e-commerce dataset
* Use `spark.read.csv()` to load all CSV files in the data directory.
* Example: `df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/", header=True, inferSchema=True)`

#### 2. Perform complex joins
* Join with another DataFrame (e.g., products, users) using different join types.
* Example: `df.join(products_df, on="product_id", how="left")`

#### 3. Calculate running totals with window functions
* Use `Window.partitionBy()` and `orderBy()` for running totals or rankings.
* Example: `from pyspark.sql.window import Window`
  `window = Window.partitionBy("user_id").orderBy("event_time")`
  `df.withColumn("running_total", F.sum("price").over(window))`

#### 4. Create derived features
* Use UDFs or built-in functions to add new columns (e.g., price category, event duration).
* Example: `df.withColumn("price_category", F.when(df.price > 100, "High").otherwise("Low"))`

In [0]:
events=spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data",header=True,inferSchema=True)
events.describe().show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Top 5 products by revenue
revenue = events.filter(F.col("event_type") == "purchase") \
    .groupBy("product_id") \
    .agg(F.sum("price").alias("revenue")) \
    .orderBy(F.desc("revenue")).limit(5)
display(revenue)

# Running total per user
window = Window.partitionBy("user_id").orderBy("event_time")
events_with_cumulative = events.withColumn("cumulative_events", F.count("*").over(window))
display(events_with_cumulative)

# Conversion rate by category
conversion = events.groupBy("category_code", "event_type").count() \
    .groupBy("category_code").pivot("event_type").sum("count") \
    .withColumn("conversion_rate", F.col("purchase")/F.col("view")*100)
display(conversion)


### Code Explanation: What Each Command Does

**Top 5 Products by Revenue**
- `events.filter(F.col("event_type") == "purchase")`: Filters the DataFrame to include only purchase events.
- `.groupBy("product_id")`: Groups the filtered data by product ID.
- `.agg(F.sum("price").alias("revenue"))`: Sums the price for each product to calculate total revenue.
- `.orderBy(F.desc("revenue")).limit(5)`: Sorts products by revenue in descending order and selects the top 5.
- `display(revenue)`: Shows the result as a table in the notebook.

**Running Total per User**
- `Window.partitionBy("user_id").orderBy("event_time")`: Defines a window for each user, ordered by event time.
- `events.withColumn("cumulative_events", F.count("*").over(window))`: Adds a column with the running count of events for each user.
- `display(events_with_cumulative)`: Displays the DataFrame with the new column.

**Conversion Rate by Category**
- `events.groupBy("category_code", "event_type").count()`: Counts the number of events for each category and event type.
- `.groupBy("category_code").pivot("event_type").sum("count")`: Pivots the event types into columns and sums counts for each category.
- `.withColumn("conversion_rate", F.col("purchase")/F.col("view")*100)`: Calculates conversion rate as purchases divided by views, multiplied by 100.
- `display(conversion)`: Shows the conversion rates by category.

---
Use these explanations to understand the logic and purpose of each step in your analysis.

### Why Am I Doing This? (Notes & Purpose)

**Top 5 Products by Revenue**
- *Purpose*: Identify which products generate the most sales revenue. This helps prioritize inventory, marketing, and business strategy.
- *Business Value*: Focus on bestsellers, optimize stock, and target promotions.

**Running Total per User**
- *Purpose*: Track user engagement over time (e.g., number of events per user). Useful for understanding user behavior and retention.
- *Business Value*: Segment users by activity, personalize offers, and improve customer experience.

**Conversion Rate by Category**
- *Purpose*: Measure how effectively product categories convert views into purchases. Reveals strengths and weaknesses in product lines.
- *Business Value*: Optimize category pages, improve product recommendations, and increase sales.

---
#### Learning Objectives
- Practice advanced PySpark: filtering, grouping, window functions, pivots, and derived metrics.
- Connect technical analysis to business questions and actionable insights.
- Build reusable code and notes for future projects or GitHub documentation.

Use these notes to understand the "why" behind each analysis and to communicate results to stakeholders or in your portfolio.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Top 5 products by revenue
# 1. Filter for purchase events only
# 2. Group by product_id
# 3. Sum price to get total revenue per product
# 4. Sort by revenue descending and take top 5
revenue = events.filter(F.col("event_type") == "purchase") \
    .groupBy("product_id") \
    .agg(F.sum("price").alias("revenue")) \
    .orderBy(F.desc("revenue")).limit(5)
display(revenue)

# Running total per user
# 1. Define a window partitioned by user_id, ordered by event_time
# 2. Add a column with the running count of events for each user
window = Window.partitionBy("user_id").orderBy("event_time")
events_with_cumulative = events.withColumn("cumulative_events", F.count("*").over(window))
display(events_with_cumulative)

# Conversion rate by category
# 1. Count events by category_code and event_type
# 2. Pivot event_type to columns (view, purchase, etc.)
# 3. Calculate conversion rate: purchases/views * 100
conversion = events.groupBy("category_code", "event_type").count() \
    .groupBy("category_code").pivot("event_type").sum("count") \
    .withColumn("conversion_rate", F.col("purchase")/F.col("view")*100)
display(conversion)


In [0]:
# List available volumes to find the correct path for your products file
spark.sql("SHOW VOLUMES IN workspace.ecommerce").show()
# After identifying the correct volume, update the file path accordingly.
# Example: products_df = spark.read.csv("/Volumes/workspace/ecommerce_data/products.csv", header=True, inferSchema=True)

In [0]:
products_df = spark.read.csv("/Volumes/ecommerce/ecommerce_data/products.csv", header=True, inferSchema=True)

In [0]:
# Join events with products_df on product_id to enrich event data with product details
joined_df = events.join(products_df, events['product_id'] == products_df['product_id'], how="inner")
#display(joined_df)
# Now you can analyze revenue, conversion, and other metrics by product attributes (name, category, etc.)

### Notes: Data Loading, Joins, and Enrichment

**1. Loading E-commerce Events Data**
- Used `spark.read.csv()` to load all event data from the Unity Catalog volume.
- Explored the schema and summary statistics to understand available columns and data types.

**2. Checking Available Volumes**
- Verified the existence of the `ecommerce_data` volume in the `ecommerce` database using `SHOW VOLUMES`.
- Ensured the correct file path for reading product details.

**3. Reading Product Details**
- Loaded product information from `products.csv` in the validated volume.
- Confirmed the file path and schema to enable joining with event data.

**4. Performing Joins for Data Enrichment**
- Used an inner join on `product_id` to combine event data with product details.
- This enriches each event with product attributes (e.g., name, category), enabling deeper analysis.

**Why is this important?**
- Data enrichment through joins allows for more meaningful business insights, such as revenue by product name or category, conversion rates by product type, and targeted marketing strategies.
- Documenting each step ensures reproducibility and clarity for future analysis or sharing on GitHub.

---
Use these notes to track your workflow, understand the value of each operation, and communicate your process to others.

In [0]:
df=[("Arpan",24,"Male",1),("Priya",25,"Female",2),("Parna",30,"Female",3)]
customers=spark.createDataFrame(df,["Name","Age","Gender","Customer_ID"])
display(customers)

In [0]:
df_n=[(1,"apple",5,1),(2,"orange",2,2),(3,"banana",12,3),(4,"grapes",20,4),(5,"mango",3,5)]
orders=spark.createDataFrame(df_n,["Product_ID","Product","Quantity","Customer_ID"])
display(orders)

In [0]:
df_joined=customers.join(orders,"Customer_ID",how="inner")
display(df_joined)

### Additional Notes: Spark DataFrames, Joins, and Best Practices

**Spark DataFrame Best Practices**
- Always inspect your DataFrame schema with `printSchema()` before performing operations.
- Use `show()` to preview data and check for unexpected nulls or outliers.
- Prefer DataFrame API over RDDs for most analytics—it's faster and easier to optimize.

**Joins: Troubleshooting and Tips**
- Ensure join keys (e.g., `product_id`, `Customer_ID`) exist in both DataFrames and have matching data types.
- For large datasets, consider broadcasting smaller DataFrames with `broadcast(df)` to speed up joins.
- Use `how="inner"` for strict matches, `how="left"` to keep all left rows, and `how="outer"` for full combinations.
- After joining, use `show()` and `count()` to verify the result and check for unexpected row loss or duplication.

**Performance and Scalability**
- Filter and select only necessary columns before joining to reduce memory usage.
- Cache intermediate results if reused, but avoid unnecessary caching.
- For repeated aggregations, consider writing results to a table or file for faster access.

**Documentation and Collaboration**
- Add markdown cells to explain your workflow, business logic, and technical choices.
- Comment your code for clarity and future reference.
- Save notebooks with descriptive titles and organize them by project or topic.

---
Use these best practices and troubleshooting tips to build robust, scalable, and well-documented Spark workflows in Databricks.